# Exploring ObsidianTools Library

This notebook demonstrates the capabilities of the `obsidiantools` library using the test vault provided in the repository. We'll explore:

1. Setting up and loading an Obsidian vault
2. Exploring vault structure
3. Working with notes and their content
4. Analyzing frontmatter
5. Exploring note links and relationships
6. Working with tags

Let's begin by importing the necessary libraries and setting up our vault.

In [1]:
import os
from pathlib import Path
from obsidiantools.api import Vault

# Set up the path to our test vault
current_dir = os.getcwd()
test_vault_path = Path(os.path.join(current_dir, 'tests', 'vault-stub'))

# Initialize the vault and connect with note content gathering
vault = Vault(test_vault_path)
vault.connect().gather()

## Exploring Vault Structure

Let's examine the structure of our vault, including the files and folders it contains.

In [2]:
# List all markdown files in the vault
print("Markdown files in the vault:")
for note_path in vault.md_file_index:
    print(f"- {os.path.basename(note_path)}")

print("\nCanvas files in the vault:")
for canvas_path in vault.canvas_file_index:
    print(f"- {os.path.basename(canvas_path)}")

# Print some basic vault statistics
print("\nVault Statistics:")
print(f"Number of markdown files: {len(vault.md_file_index)}")
print(f"Number of canvas files: {len(vault.canvas_file_index)}")

Markdown files in the vault:
- Isolated note
- Sussudio
- rich
- Isolated note
- Causam mihi
- Vulnera ubera
- Ne fuit
- Brevissimus moenia
- Alimenta

Canvas files in the vault:
- Crazy wall.canvas
- Crazy wall 2.canvas

Vault Statistics:
Number of markdown files: 9
Number of canvas files: 2


## Working with Notes and Frontmatter

Let's examine the content and frontmatter of some notes in our vault. We'll look at:
1. Reading note content
2. Extracting frontmatter
3. Analyzing note metadata

In [3]:
# Get frontmatter using vault's built-in front_matter_index
print("Frontmatter for Sussudio:")
sussudio_frontmatter = vault.get_front_matter('Sussudio')
if sussudio_frontmatter:
    for key, value in sussudio_frontmatter.items():
        print(f"{key}: {value}")

# Get note content using vault's source_text and readable_text features
print("\nSource text (first few lines):")
source_text = vault.get_source_text('Sussudio')
print("\n".join(source_text.split("\n")[:5]))

print("\nReadable text (processed for better readability):")
readable_text = vault.get_readable_text('Sussudio')
print("\n".join(readable_text.split("\n")[:5]))

Frontmatter for Sussudio:
title: Sussudio
artist: Phil Collins
category: music
year: 1985
url: https://www.discogs.com/Phil-Collins-Sussudio/master/106239
references: [[['American Psycho (film)']], 'Polka Party!']
chart_peaks: [{'US': 1}, {'UK': 12}]

Source text (first few lines):
# Sussudio

Another word with absolutely no meaning 😄

This will be a note inside the vault dir. Others will be lipsum in a subdirectory.

Readable text (processed for better readability):
# Sussudio

Another word with absolutely no meaning 😄

This will be a note inside the vault dir. Others will be lipsum in a subdirectory.


## Exploring Note Links and References

Let's examine how notes are connected to each other through links and references. We'll analyze both internal links and backlinks.

In [4]:
# Let's analyze different types of links in notes

# Analyze Sussudio.md
print("Links in Sussudio.md:")
print("\nWikilinks:")
wikilinks = vault.get_wikilinks('Sussudio')
for link in wikilinks:
    print(f"- {link}")

print("\nBacklinks (notes that link to Sussudio):")
backlinks = vault.get_backlinks('Sussudio')
for link in backlinks:
    print(f"- {link}")

print("\nEmbedded files:")
embedded_files = vault.get_embedded_files('Sussudio')
for file in embedded_files:
    print(f"- {file}")

print("\nMarkdown links:")
md_links = vault.get_md_links('Sussudio')
for link in md_links:
    print(f"- {link}")

# Analyze links in another note (Vulnera ubera)
print("\nAnalyzing Vulnera ubera:")
print("\nWikilinks:")
vulnera_wikilinks = vault.get_wikilinks('Vulnera ubera')
for link in vulnera_wikilinks:
    print(f"- {link}")

# You can also get link counts
print("\nLink statistics for Sussudio:")
wikilink_counts = vault.get_wikilink_counts('Sussudio')
for target, count in wikilink_counts.items():
    print(f"Links to {target}: {count}")

backlink_counts = vault.get_backlink_counts('Sussudio')
for source, count in backlink_counts.items():
    print(f"Links from {source}: {count}")

Links in Sussudio.md:

Wikilinks:
- American Psycho (film)

Backlinks (notes that link to Sussudio):

Embedded files:
- Sussudio.mp3
- 1999.flac

Markdown links:

Analyzing Vulnera ubera:

Wikilinks:
- Caelum
- Tarpeia
- Vita

Link statistics for Sussudio:
Links to American Psycho (film): 1


## Working with Tags

Let's analyze the tags used across notes in the vault.

In [5]:
from obsidiantools.md_utils import get_tags

# Get tags using the vault's built-in tags_index
print("All unique tags in the vault from the vault.tags_index:")
all_tags = set()
for note_tags in vault.tags_index.values():
    all_tags.update(note_tags)

for tag in sorted(all_tags):
    print(f"- {tag}")

print("\nFiles with their tags:")
for note_name, tags in vault.tags_index.items():
    if tags:  # Only show files that have tags
        print(f"\n{note_name}:")
        for tag in sorted(tags):
            print(f"  - {tag}")

# You can also get tags for a specific note
print("\nDemonstrating get_tags() for a specific note (Sussudio.md):")
sussudio_tags = get_tags(Path(os.path.join(test_vault_path, 'Sussudio.md')), show_nested=True)
print("\nTags in Sussudio.md (including nested tags):")
for tag in sorted(sussudio_tags):
    print(f"- {tag}")

All unique tags in the vault from the vault.tags_index:
- y-1982
- y1982
- y2000
- y_1982

Files with their tags:

Sussudio:
  - y-1982
  - y1982
  - y1982
  - y2000
  - y_1982

Demonstrating get_tags() for a specific note (Sussudio.md):

Tags in Sussudio.md (including nested tags):
- y-1982
- y1982
- y1982/sep
- y2000/party-over/oops/out-of-time
- y_1982


## Working with Properties

Let's explore how to work with Obsidian properties, which can be defined either in the frontmatter or inline in the note. Properties in Obsidian are a powerful way to add structured metadata to your notes.

In [6]:
# Let's look at properties in the test vault
from pathlib import Path
import json
from datetime import date, datetime

class DateTimeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (date, datetime)):
            return obj.isoformat()
        return super().default(obj)

def display_properties_for_note(vault, note_name):
    """Helper function to display properties for a note"""
    props = vault.get_properties(note_name)
    if props:
        print(f"\nProperties for {note_name}:")
        for key, value in props.items():
            print(f"  {key}: {value}")
    else:
        print(f"\nNo properties found for {note_name}")

# Get properties for all notes in the vault
print("Properties Overview:")
for note_name in vault.md_file_index:
    display_properties_for_note(vault, note_name)

# Get properties index (all properties across all notes)
print("\nProperties Index:")
properties_index = vault.get_properties_index()
print(json.dumps(properties_index, indent=2, cls=DateTimeEncoder))

# Example: Get a specific property from a note
print("\nExample: Getting specific properties")
status = vault.get_property('Sussudio', 'status')
if status:
    print(f"Status of Sussudio: {status}")

# Example: Working with property types
print("\nExample: Different property types")
test_files = Path(os.path.join(current_dir, 'tests', 'general', 'property-tests'))
for note_path in test_files.glob('*.md'):
    note_name = note_path.stem
    props = vault.get_properties(note_name)
    if props:
        print(f"\n{note_name}:")
        for key, value in props.items():
            print(f"  {key} ({type(value).__name__}): {value}")

Properties Overview:

No properties found for Isolated note

Properties for Sussudio:
  title: Sussudio
  artist: Phil Collins
  category: music
  year: 1985
  url: https://www.discogs.com/Phil-Collins-Sussudio/master/106239
  references: [[['American Psycho (film)']], 'Polka Party!']
  chart_peaks: [{'US': 1}, {'UK': 12}]

Properties for rich:
  title: Rich Properties Example
  date: 2025-07-17
  time: 2025-07-17 15:30:00
  status: In Progress
  tags: ['obsidian', 'test', 'properties']
  mixed bag: [1, 'two', True, None, '', 'None']
  nested: {'key1': 'value1', 'key2': {'subkey1': 'nested value', 'subkey2': 'another value'}}
  priority: High
  links: [[['another-note']], [['yet-another-note|custom link text']]]
  prop with spaces: special value
  property:with:colons: complex:value:here
  number: 42
  empty: 

No properties found for lipsum/Isolated note

Properties for Causam mihi:
  title: Causam mihi
  author: Ovid
  category: literature
  year: 8
  language: la
  description: \{\{

Key features of the properties implementation:

1. **Property Sources**
   - YAML frontmatter at the start of files
   - Inline properties in the format `property:: value`
   - Inline array properties using `[value1, value2]` syntax

2. **Property Types**
   - String values (with special character support)
   - Arrays
   - Numbers
   - Dates (automatically converted to ISO format)

3. **Special Features**
   - Property overriding (inline properties override frontmatter)
   - Special character handling in property keys and values
   - Quote-preserving property values
   - Array type support

Let's now explore some practical examples of working with properties:

In [7]:
# Advanced Property Examples

def find_notes_by_property(vault, property_name, property_value=None):
    """Find notes that have a specific property, optionally matching a value"""
    matching_notes = []
    for note_name in vault.md_file_index:
        props = vault.get_properties(note_name)
        if not props:
            continue
            
        if property_value is None:
            # Just check if property exists
            if property_name in props:
                matching_notes.append(note_name)
        else:
            # Check if property matches value
            if props.get(property_name) == property_value:
                matching_notes.append(note_name)
    
    return matching_notes

def get_property_statistics(vault):
    """Get statistics about property usage in the vault"""
    stats = {
        'total_notes': len(vault.md_file_index),
        'notes_with_properties': 0,
        'property_counts': {},
        'property_types': {},
        'array_properties': set(),
    }
    
    for note_name in vault.md_file_index:
        props = vault.get_properties(note_name)
        if props:
            stats['notes_with_properties'] += 1
            for key, value in props.items():
                stats['property_counts'][key] = stats['property_counts'].get(key, 0) + 1
                value_type = type(value).__name__
                if value_type not in stats['property_types']:
                    stats['property_types'][value_type] = set()
                stats['property_types'][value_type].add(key)
                if isinstance(value, list):
                    stats['array_properties'].add(key)
    
    return stats

# Example usage
print("Finding notes by property:")
status_notes = find_notes_by_property(vault, 'status', 'In Progress')
print("\nNotes with status='In Progress':")
for note in status_notes:
    print(f"- {note}")

print("\nProperty Statistics:")
stats = get_property_statistics(vault)
print(f"\nTotal notes: {stats['total_notes']}")
print(f"Notes with properties: {stats['notes_with_properties']}")
print("\nProperty usage counts:")
for prop, count in stats['property_counts'].items():
    print(f"- {prop}: {count} notes")
print("\nProperty types:")
for type_name, props in stats['property_types'].items():
    print(f"\n{type_name} properties:")
    for prop in props:
        print(f"- {prop}")
print("\nArray properties:")
for prop in stats['array_properties']:
    print(f"- {prop}")

Finding notes by property:

Notes with status='In Progress':
- rich

Property Statistics:

Total notes: 9
Notes with properties: 3

Property usage counts:
- title: 3 notes
- artist: 1 notes
- category: 2 notes
- year: 2 notes
- url: 1 notes
- references: 1 notes
- chart_peaks: 1 notes
- date: 1 notes
- time: 1 notes
- status: 1 notes
- tags: 1 notes
- mixed bag: 1 notes
- nested: 1 notes
- priority: 1 notes
- links: 1 notes
- prop with spaces: 1 notes
- property:with:colons: 1 notes
- number: 1 notes
- empty: 1 notes
- author: 1 notes
- language: 1 notes
- description: 1 notes

Property types:

str properties:
- artist
- category
- prop with spaces
- description
- priority
- title
- status
- language
- url
- author
- property:with:colons
- empty
- number

int properties:
- year

list properties:
- chart_peaks
- references
- links
- tags
- mixed bag

date properties:
- date

datetime properties:
- time

dict properties:
- nested

Array properties:
- chart_peaks
- references
- links
- ta

## Summary

In this notebook, we've explored the main capabilities of the `obsidiantools` library:

1. **Vault Structure**: We learned how to initialize a vault and list its markdown and canvas files
2. **Note Content**: We demonstrated how to read and parse note content, including frontmatter
3. **Links and References**: We explored how to extract and analyze internal links between notes
4. **Tags**: We showed how to extract and analyze tags from both note content and frontmatter

The library provides a solid foundation for programmatically analyzing and working with Obsidian vaults. Some key features we discovered:

- Support for both markdown (.md) and canvas (.canvas) files
- YAML frontmatter parsing
- Wiki-style link extraction
- Tag analysis from both content and frontmatter

You can use these tools to build more complex analyses of your Obsidian vault, such as:
- Creating network graphs of note relationships
- Analyzing tag usage patterns
- Extracting structured data from frontmatter
- Building custom search and navigation tools

## Additional Vault Methods Reference

Let's explore all the available methods in the Vault class that can be useful for automation and analysis.

### Core Methods

1. **Setup Methods**
   - `connect(show_nested_tags=False, attachments=False)`: Connects notes in a graph structure
   - `gather(tags=None)`: Gathers text content of notes for analysis

2. **Note Access Methods**
   - `get_source_text(note_name)`: Get raw source text of a note
   - `get_readable_text(note_name)`: Get processed text optimized for reading/analysis
   - `get_front_matter(note_name)`: Get note's frontmatter
   - `get_tags(note_name)`: Get tags from a note

3. **Link Analysis Methods**
   - `get_wikilinks(note_name)`: Get wiki-style links from a note
   - `get_backlinks(note_name)`: Get notes that link to this note
   - `get_wikilink_counts(note_name)`: Get count of outgoing links
   - `get_backlink_counts(note_name)`: Get count of incoming links
   - `get_embedded_files(file_name)`: Get list of embedded files
   - `get_md_links(file_name)`: Get markdown-style links

4. **Graph and Network Methods**
   - `get_note_metadata()`: Get DataFrame with note statistics
   - `get_media_file_metadata()`: Get DataFrame with media file statistics
   - `get_canvas_file_metadata()`: Get DataFrame with canvas file statistics
   - `get_all_file_metadata()`: Get combined DataFrame of all file types

### Important Properties

1. **File Indices**
   - `md_file_index`: Dictionary of markdown files
   - `canvas_file_index`: Dictionary of canvas files
   - `media_file_index`: Dictionary of media files

2. **Content Indices**
   - `front_matter_index`: Dictionary of all notes' frontmatter
   - `tags_index`: Dictionary of all notes' tags
   - `source_text_index`: Dictionary of all notes' source text
   - `readable_text_index`: Dictionary of all notes' readable text

3. **Link Indices**
   - `wikilinks_index`: Dictionary of all wiki-style links
   - `backlinks_index`: Dictionary of all backlinks
   - `embedded_files_index`: Dictionary of all embedded files
   - `md_links_index`: Dictionary of all markdown links

4. **Special Note Lists**
   - `isolated_notes`: Notes without any connections
   - `nonexistent_notes`: Referenced notes that don't exist yet

5. **Graph Access**
   - `graph`: NetworkX graph object representing the vault

Let's demonstrate some of these additional features with examples.

In [8]:
import pandas as pd
import networkx as nx

# 1. Get metadata about all notes in a DataFrame
note_metadata = vault.get_note_metadata()
print("Note Metadata Overview:")
print(note_metadata.head())

# 2. Find isolated notes (notes without connections)
print("\nIsolated Notes (no connections):")
for note in vault.isolated_notes:
    print(f"- {note}")

# 3. Find nonexistent notes (referenced but don't exist)
print("\nNonexistent Notes (referenced but missing):")
for note in vault.nonexistent_notes:
    print(f"- {note}")

# 4. Access the graph for network analysis
graph = vault.graph
print("\nGraph Statistics:")
print(f"Number of nodes: {graph.number_of_nodes()}")
print(f"Number of edges: {graph.number_of_edges()}")

# 5. Show all unique wikilinks in the vault
print("\nAll Unique Wikilinks:")
all_wikilinks = set()
for links in vault.wikilinks_index.values():
    all_wikilinks.update(links)
for link in sorted(all_wikilinks):
    print(f"- {link}")

# 6. Show media file statistics
media_metadata = vault.get_media_file_metadata()
print("\nMedia File Statistics:")
if not media_metadata.empty:
    print(media_metadata.head())
else:
    print("No media files found")

# 7. Show canvas file information
canvas_metadata = vault.get_canvas_file_metadata()
print("\nCanvas File Statistics:")
if not canvas_metadata.empty:
    print(canvas_metadata.head())
else:
    print("No canvas files found")

Note Metadata Overview:
                       rel_filepath  \
note                                  
American Psycho (film)          NaN   
Amor                            NaN   
Sussudio                Sussudio.md   
Aetna                           NaN   
Aras Teucras                    NaN   

                                                             abs_filepath  \
note                                                                        
American Psycho (film)                                                NaN   
Amor                                                                  NaN   
Sussudio                /Users/veethahavya/Projects/obsidiantools/test...   
Aetna                                                                 NaN   
Aras Teucras                                                          NaN   

                        note_exists  n_backlinks  n_wikilinks  n_tags  \
note                                                                    
American Psycho 

### Practical Automation Examples

Here are some practical ways to use these methods for vault automation:

In [9]:
# Example 1: Find notes that need attention (no tags, no links)
def find_notes_needing_attention():
    notes_to_review = []
    for note_name in vault.md_file_index:
        # Check if note has no tags
        has_no_tags = note_name not in vault.tags_index or not vault.tags_index[note_name]
        # Check if note has no outgoing links
        has_no_links = note_name not in vault.wikilinks_index or not vault.wikilinks_index[note_name]
        # Check if note has no backlinks
        has_no_backlinks = note_name not in vault.backlinks_index or not vault.backlinks_index[note_name]
        
        if has_no_tags and (has_no_links or has_no_backlinks):
            notes_to_review.append(note_name)
    
    return notes_to_review

# Example 2: Create a reference network for a specific topic
def create_topic_network(topic_tag):
    relevant_notes = []
    for note, tags in vault.tags_index.items():
        if topic_tag in tags:
            relevant_notes.append(note)
    
    topic_network = {}
    for note in relevant_notes:
        connections = {
            'outgoing': vault.get_wikilinks(note),
            'incoming': vault.get_backlinks(note)
        }
        topic_network[note] = connections
    
    return topic_network

# Example 3: Analyze note complexity
def analyze_note_complexity():
    complexity_metrics = {}
    for note_name in vault.md_file_index:
        if note_name in vault.source_text_index:
            metrics = {
                'word_count': len(vault.get_readable_text(note_name).split()),
                'link_count': len(vault.get_wikilinks(note_name)),
                'backlink_count': len(vault.get_backlinks(note_name)),
                'tag_count': len(vault.tags_index.get(note_name, [])),
                'has_frontmatter': bool(vault.get_front_matter(note_name))
            }
            complexity_metrics[note_name] = metrics
    
    return pd.DataFrame.from_dict(complexity_metrics, orient='index')

# Let's try these functions
print("Notes Needing Attention:")
for note in find_notes_needing_attention():
    print(f"- {note}")

print("\nNetwork for music-related notes:")
music_network = create_topic_network('music')
for note, connections in music_network.items():
    print(f"\n{note}:")
    print("  Outgoing links:", connections['outgoing'])
    print("  Incoming links:", connections['incoming'])

print("\nNote Complexity Analysis:")
complexity_df = analyze_note_complexity()
print(complexity_df.head())

Notes Needing Attention:
- Isolated note
- rich
- lipsum/Isolated note
- Vulnera ubera
- Alimenta

Network for music-related notes:

Note Complexity Analysis:
                      word_count  link_count  backlink_count  tag_count  \
Isolated note                  9           0               0          0   
Sussudio                      71           1               0          5   
rich                          57           0               0          0   
lipsum/Isolated note          30           0               0          0   
Causam mihi                  200           4               1          0   

                      has_frontmatter  
Isolated note                   False  
Sussudio                         True  
rich                             True  
lipsum/Isolated note            False  
Causam mihi                      True  


### Advanced Graph Analysis

Since we have access to the full NetworkX graph of the vault, we can perform sophisticated network analysis:

In [10]:
# Get the graph
G = vault.graph

# 1. Calculate basic centrality measures
centrality = {
    'degree': nx.degree_centrality(G),
    'in_degree': nx.in_degree_centrality(G),
    'out_degree': nx.out_degree_centrality(G)
}

# Create a DataFrame of centrality measures
centrality_df = pd.DataFrame(centrality)
print("Note Centrality Analysis (top 5 by degree):")
print(centrality_df.sort_values('degree', ascending=False).head())

# 2. Find weakly connected components (groups of connected notes)
components = list(nx.weakly_connected_components(G))
print("\nConnected Components:")
for i, component in enumerate(components, 1):
    print(f"Component {i}: {len(component)} notes")
    print(f"Notes: {', '.join(sorted(component))}")

# 3. Calculate shortest paths between notes
def find_note_connections(start_note, max_distance=2):
    paths = {}
    try:
        for target in G.nodes():
            if target != start_note:
                try:
                    path = nx.shortest_path(G, start_note, target)
                    if len(path) - 1 <= max_distance:  # -1 because path includes start node
                        paths[target] = path
                except nx.NetworkXNoPath:
                    continue
    except nx.NetworkXError:
        return {}
    return paths

# Example: Find connections from Sussudio
print("\nConnections from Sussudio (up to 2 links away):")
sussudio_connections = find_note_connections('Sussudio')
for target, path in sussudio_connections.items():
    print(f"To {target}: {' -> '.join(path)}")

# 4. Basic graph statistics
print("\nGraph Statistics:")
print(f"Number of notes (nodes): {G.number_of_nodes()}")
print(f"Number of links (edges): {G.number_of_edges()}")
print(f"Average out-degree: {sum(dict(G.out_degree()).values()) / G.number_of_nodes():.2f}")
print(f"Notes with no outgoing links: {sum(1 for _, d in G.out_degree() if d == 0)}")
print(f"Notes with no incoming links: {sum(1 for _, d in G.in_degree() if d == 0)}")

Note Centrality Analysis (top 5 by degree):
                    degree  in_degree  out_degree
Alimenta              0.60       0.00        0.60
Ne fuit               0.40       0.10        0.30
Causam mihi           0.25       0.05        0.20
Bacchus               0.25       0.25        0.00
Brevissimus moenia    0.20       0.05        0.15

Connected Components:
Component 1: 1 notes
Notes: Isolated note
Component 2: 2 notes
Notes: American Psycho (film), Sussudio
Component 3: 1 notes
Notes: rich
Component 4: 1 notes
Notes: lipsum/Isolated note
Component 5: 16 notes
Notes: Aetna, Alimenta, Amor, Aras Teucras, Bacchus, Brevissimus moenia, Caelum, Causam mihi, Dives, Manus, Ne fuit, Tarpeia, Tydides, Virtus, Vita, Vulnera ubera

Connections from Sussudio (up to 2 links away):
To American Psycho (film): Sussudio -> American Psycho (film)

Graph Statistics:
Number of notes (nodes): 21
Number of links (edges): 29
Average out-degree: 1.38
Notes with no outgoing links: 15
Notes with no incom